In [0]:
%run "/Workspace/Users/gowtham.thinqnxt@gmail.com/media_entertainment/service principle"

In [0]:
BASE_PATH = "abfss://source@babustorage01.dfs.core.windows.net/media_lakehouse"


In [0]:
!pip install faker
from pyspark.sql.functions import *
from pyspark.sql.types import *
from faker import Faker
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random
fake = Faker()

In [0]:
USERS = [f"U{str(i).zfill(3)}" for i in range(1, 11)]
CONTENTS = [f"C{str(i).zfill(3)}" for i in range(1, 6)]
CAMPAIGNS = ["CMP001", "CMP002"]
SUBSCRIPTIONS = {u: f"S{u[1:]}" for u in USERS}
PLATFORMS = ["Twitter","Facebook","Instagram","LinkedIn"]


In [0]:
#content schema
content_schema = StructType([
    StructField("content_id", StringType()),
    StructField("title", StringType()),
    StructField("content_type", StringType()),
    StructField("genre", StringType()),
    StructField("language", StringType()),
    StructField("release_date", DateType()),
    StructField("duration_sec", LongType()),
    StructField("publisher", StringType()),
    StructField("tags", ArrayType(StringType())),
    StructField("last_updated", TimestampType())
])


In [0]:
content_data = [
    (
        c,
        fake.sentence(3),
        random.choice(["MOVIE","SERIES"]),
        random.choice(["Action","Drama","Kids"]),
        "EN",
        fake.date_between("-2y","today"),
        random.randint(1800,7200),
        fake.company(),
        [fake.word(), fake.word()],
        datetime.now()
    )
    for c in CONTENTS
]

spark.createDataFrame(content_data, content_schema) \
    .write.mode("append") \
    .parquet(f"{BASE_PATH}/content_raw")


In [0]:
#user profile
user_schema = StructType([
    StructField("user_id", StringType()),
    StructField("signup_ts", TimestampType()),
    StructField("country", StringType()),
    StructField("region", StringType()),
    StructField("language", StringType()),
    StructField("age_band", StringType()),
    StructField("marketing_opt_in", BooleanType()),
    StructField("last_updated", TimestampType())
])


In [0]:
user_data = [
    (
        u,
        fake.date_time_between("-1y","now"),
        random.choice(["IN","US"]),
        random.choice(["AP","TS","CA"]),
        "EN",
        random.choice(["18-25","26-35","36-45"]),
        random.choice([True,False]),
        datetime.now()
    )
    for u in USERS
]

spark.createDataFrame(user_data, user_schema) \
    .write.mode("append") \
    .parquet(f"{BASE_PATH}/user_profile_raw")


In [0]:
#subscription
subscription_schema = StructType([
    StructField("subscription_id", StringType()),
    StructField("user_id", StringType()),
    StructField("plan_name", StringType()),
    StructField("price_usd", DoubleType()),
    StructField("status", StringType()),
    StructField("start_date", DateType()),
    StructField("end_date", DateType()),
    StructField("cancel_reason", StringType()),
    StructField("last_updated", TimestampType())
])


In [0]:
from datetime import timedelta

subscription_data = []

for u in USERS:
    status = random.choice(["ACTIVE", "CANCELLED"])

    start_date = fake.date_between("-1y", "-10d")

    if status == "CANCELLED":
        end_date = fake.date_between(start_date, "today")
        cancel_reason = random.choice(["PRICE", "CONTENT", "COMPETITOR"])
    else:
        end_date = None
        cancel_reason = None

    subscription_data.append((
        SUBSCRIPTIONS[u],
        u,
        random.choice(["BASIC", "PREMIUM"]),
        random.choice([4.99, 9.99]),
        status,
        start_date,      # ✅ never NULL
        end_date,        # ✅ conditional
        cancel_reason,
        datetime.now()
    ))

spark.createDataFrame(subscription_data, subscription_schema) \
    .write.mode("append") \
    .parquet(f"{BASE_PATH}/subscription_raw")


In [0]:
#payment raw
payment_schema = StructType([
    StructField("payment_id", StringType()),
    StructField("subscription_id", StringType()),
    StructField("user_id", StringType()),
    StructField("payment_ts", TimestampType()),
    StructField("amount_usd", DoubleType()),
    StructField("status", StringType()),
    StructField("provider", StringType())
])


In [0]:
from datetime import datetime, timedelta

now = datetime.utcnow()   # or datetime.now() if you prefer local time

# Find current 6-hour window start
batch_hour = (now.hour // 6) * 6

window_start = now.replace(
    hour=batch_hour,
    minute=0,
    second=0,
    microsecond=0
)

window_end = window_start + timedelta(hours=6)


payment_data = [
    (
        f"P{fake.random_int(100000,999999)}",
        SUBSCRIPTIONS[u],
        u,
        fake.date_time_between(window_start, window_end),
        random.choice([4.99,9.99]),
        "SUCCESS",
        random.choice(["STRIPE","PAYPAL"])
    )
    for u in USERS
]

spark.createDataFrame(payment_data, payment_schema) \
    .write.mode("append") \
    .parquet(f"{BASE_PATH}/payment_raw")


In [0]:
#social signals
social_schema = StructType([
    StructField("signal_id", StringType()),
    StructField("platform", StringType()),
    StructField("content_id", StringType()),
    StructField("mention_ts", TimestampType()),
    StructField("sentiment", DoubleType()),
    StructField("mentions", IntegerType()),
    StructField("engagement", IntegerType())
])


In [0]:
from pyspark.sql.functions import round

social_data = [
    (
        f"SIG{fake.random_int(100000,999999)}",
        random.choice(PLATFORMS),
        random.choice(CONTENTS),
        fake.date_time_between(window_start, window_end),
        float(f"{random.uniform(-1,1):.2f}"),
        random.randint(1,300),
        random.randint(50,5000)
    )
    for _ in range(10)
]

spark.createDataFrame(social_data, social_schema) \
    .write \
    .mode("append") \
    .parquet(f"{BASE_PATH}/social_signals_raw")



In [0]:
#user events raw
events_schema = StructType([
    StructField("event_id", StringType()),
    StructField("user_id", StringType()),
    StructField("session_id", StringType()),
    StructField("content_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_ts", TimestampType()),
    StructField("position_sec", IntegerType()),
    StructField("watch_time_sec", IntegerType()),
    StructField("query", StringType()),
    StructField("clicked_content_id", StringType()),
    StructField("device_type", StringType()),
    StructField("os", StringType()),
    StructField("app_version", StringType()),
    StructField("platform", StringType()),
    StructField("country", StringType()),
    StructField("region", StringType()),
    StructField("referrer", StringType()),
    StructField("error_code", StringType())
])


In [0]:
EVENT_TYPES = ["PLAY", "PAUSE", "STOP", "SEARCH","COMPLETE","SEARCH"]
ERROR_CODES = ["E101", "E202", "E303", "NONE"]

events_data = [
    (
        f"E{fake.random_int(100000,999999)}",           # event_id
        random.choice(USERS),                            # user_id
        f"SESS{fake.random_int(1000,9999)}",             # session_id
        random.choice(CONTENTS),                         # content_id
        random.choice(EVENT_TYPES),                      # event_type
        fake.date_time_between(window_start, window_end),# event_ts
        random.randint(0, 1800),                          # position_sec
        random.randint(0, 3600),                          # watch_time_sec
        fake.word(),                                     # query (NON-NULL)
        random.choice(CONTENTS),                         # clicked_content_id
        random.choice(["MOBILE","WEB","TV"]),             # device_type
        random.choice(["ANDROID","IOS","WINDOWS"]),       # os
        "1.0.0",                                         # app_version
        random.choice(["APP","WEB","TV"]),                # platform
        random.choice(["IN","US"]),                       # country
        random.choice(["AP","TS","CA"]),                  # region
        random.choice(["HOME","SEARCH","RECO"]),          # referrer
        random.choice(ERROR_CODES)                        # error_code (NON-NULL)
    )
    for _ in range(10)
]


spark.createDataFrame(events_data, events_schema) \
    .write.mode("append") \
    .parquet(f"{BASE_PATH}/user_events_raw")


In [0]:
#ad events raw
from pyspark.sql.types import *

ad_events_schema = StructType([
    StructField("ad_event_id", StringType()),
    StructField("user_id", StringType()),
    StructField("session_id", StringType()),
    StructField("content_id", StringType()),
    StructField("ad_id", StringType()),
    StructField("campaign_id", StringType()),
    StructField("placement", StringType()),
    StructField("ad_format", StringType()),
    StructField("event_type", StringType()),
    StructField("event_ts", TimestampType()),
    StructField("revenue_usd", DoubleType())
])


In [0]:
ad_events_data = [
    (
        f"AE{fake.random_int(100000,999999)}",   # ad_event_id
        random.choice(USERS),                    # user_id
        f"SESS{fake.random_int(1000,9999)}",     # session_id
        random.choice(CONTENTS),                 # content_id
        f"AD{fake.random_int(100,999)}",         # ad_id
        random.choice(CAMPAIGNS),                # campaign_id
        random.choice(["PRE_ROLL","MID_ROLL"]),  # placement
        random.choice(["VIDEO","BANNER"]),       # ad_format
        random.choice(["IMPRESSION","CLICK"]),   # event_type
        fake.date_time_between(window_start, window_end),  # event_ts
        float(f"{random.uniform(0.001, 0.02):.6f}")
    # revenue_usd
    )
    for _ in range(10)
]

spark.createDataFrame(ad_events_data, ad_events_schema) \
    .write \
    .mode("append") \
    .parquet(f"{BASE_PATH}/ad_events_raw")



In [0]:
#campaign raw
from pyspark.sql.types import *

campaign_schema = StructType([
    StructField("campaign_id", StringType()),
    StructField("advertiser", StringType()),
    StructField("campaign_name", StringType()),
    StructField("objective", StringType()),
    StructField("start_date", DateType()),
    StructField("end_date", DateType()),
    StructField("last_updated", TimestampType())
])


In [0]:
campaign_data = [
    (
        cid,                                # campaign_id (shared key)
        fake.company(),                     # advertiser
        fake.catch_phrase(),                # campaign_name
        random.choice(["AWARENESS","CONVERSION","ENGAGEMENT"]),
        fake.date_between("-30d","today"),  # start_date
        fake.date_between("today","+30d"),  # end_date
        datetime.now()                      # last_updated
    )
    for cid in CAMPAIGNS
]

# ensure exactly 10 rows (repeat campaigns if needed)
campaign_data = (campaign_data * 5)[:10]
spark.createDataFrame(campaign_data, campaign_schema) \
    .write \
    .mode("append") \
    .parquet(f"{BASE_PATH}/campaign_raw")

